In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#import tf
import torch
import time
import math
from IPython.display import clear_output
import os
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv
import tensorflow as tf

# torch.compat.v1.enable_eager_execution()

In [ ]:
### Solver ###
class TGD_class():
    #Initial parameters
    def __init__(self, K, dt,zeta, sigma,D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros):
        self.K = K.type(torch.complex64) 
        self.dt = dt
        self.MAX_ITER = MAX_ITER
        self.unity = unity.to('cuda')
        self.zeros=zeros.to('cuda')
        #scale = torch.dtypes.cast(Get_Scale(K,N), dtype='complex64')
        self.eps_coeff = eps_coeff
        self.rho_thr_coeff = rho_thr_coeff
        self.gamma_init_coeff = gamma_init_coeff #0.2*scale
        self.N=N
        self.attempt_num=attempt_num
        self.pi = np.pi
        self.zeta=zeta
        self.sigma=sigma
        self.D_coeff=D_coeff
        self.scale = torch.max(torch.einsum('ijk->i', torch.abs(self.K)))
#     def Get_Scale(self):
#         isum=torch.dtypes.cast(torch.einsum('ijk->i', torch.abs(self.K)), dtype='float')
#         scale = torch.reduce_max(isum)
#         return scale
    
    def GD_Solver(self, phi_old, phi_new, gamma, eps, rho_thr):
        Density = torch.abs(phi_old)**2
#         print(Density.size(), unity.size(), self.zeros.size(), rho_thr.size())
        D = self.D_coeff*torch.maximum(self.unity-Density/rho_thr,self.zeros) 
        Noise = self.sigma * 1j * math.pi * torch.rand(self.N, self.attempt_num, device = torch.device('cuda'))
        Noise = torch.exp(Noise)               
        dphi = torch.multiply(gamma - Density , phi_old)
        #for y in range(N):
        
        
        
        
        dphi-=self.zeta/self.N*torch.sqrt(rho_thr)*torch.einsum('ijl,js,ls->is', self.K, phi_old, torch.conj(phi_old)) #torch.outer(phi_old,torch.conj(phi_old))
        phi_new = phi_old + self.dt * (dphi + torch.multiply(D,Noise))  
        gamma = gamma + self.dt * eps * (rho_thr * self.unity - Density)
        return phi_new, gamma
    
    
    
    def GD_eval(self):
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        phi_1 = torch.zeros(self.N,self.attempt_num, dtype = torch.complex64, device = torch.device('cuda'))
        phi_2 = phi_1.clone()
        
        gamma = -gamma_init * torch.ones((self.N,self.attempt_num), device = torch.device('cuda'))
        
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K.to(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    
    
    
    
    
    def GD_eval_cc(self, phi_1, gamma, K1,K):
        self.K=K1
        self.K_en=K
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        #gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        #phi_1 = torch.zeros((self.N,self.attempt_num), dtype='complex64')
        phi_2 = phi_1.clone()
        #gamma = -gamma_init * torch.ones((self.N,self.attempt_num), dtype='complex64')
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K_en.type(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    

In [ ]:
class TGDInterface(mli.Interface):
        def __init__(self, tgd, attempt_num, clip_percents): 
            super(TGDInterface, self).__init__()
            self.tgd=tgd
            self.attempt_num = attempt_num
            self.tgd.GD_eval()
            #opt_norm=sess.run(tgd.lastcuts)
            self.mean = np.mean(np.array(tgd.lastcuts.cpu()))
            self.scaleen = np.min(np.array(tgd.lastcuts.cpu()))#+self.perc
            self.minimum = np.min(np.array(tgd.lastcuts.cpu()))
            #self.TGD.pump(O,S,D,N, Jmax)

            #self.TGD.pump(O,S,D,N, Jmax)



        def get_sim_params(self, packed_params):
            num_single = 7
            logsig, logdt, logzeta, logeps, logrho, loggamma, logD  = packed_params[:num_single]

            sigma = 10**logsig
            dt = 10**logdt
            zeta = 10**logzeta
            eps_coeff=10**logeps
            rho_thr_coeff=10**logrho
            gamma_init_coeff=10**loggamma
            D_coeff=10**logD
            return sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff


        def get_next_cost_dict(self, params_dict):
            params = params_dict['params']

            sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff = self.get_sim_params(params)

            print('\nTGD:\nsigma={}\ndt={}\nzeta={}\neps_coeff={}\nrho_thr_coeff={}\ngamma_init_coeff={}\nD_coeff={}\n'.format(sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff))

            self.tgd.sigma = sigma
            self.tgd.dt = dt
            self.tgd.zeta = zeta
            self.tgd.eps_coeff=eps_coeff
            self.tgd.rho_thr_coeff=rho_thr_coeff
            self.tgd.gamma_init_coeff=gamma_init_coeff
            self.tgd.D_coeff=D_coeff

            #self.TGD.rate=rate
            self.tgd.GD_eval()
            #H_eval=sess.run(self.tgd.lastcuts)
            cost_perc = min([np.mean(np.array(self.tgd.lastcuts.cpu()))/np.abs(self.mean),1]) #np.min(self.TGD.lastcuts)
            cost_min = min([np.min(np.array(self.tgd.lastcuts.cpu()))/np.abs(self.scaleen), 1]) 
            cost=cost_min+0.5*cost_perc
            #cost=min([np.min(self.TGD.lastcuts.cpu())/self.minimum, 1])
            uncer = 0
            bad = False
            print('Initial perc', self.mean)
#             target_perc=self.mean
#             target_scale=self.scale_en
            print('Final perc', np.mean(np.array(self.tgd.lastcuts.cpu())))
            print('Final energy', np.min(np.array(self.tgd.lastcuts.cpu())))
            cost_dict = {'cost':cost, 'uncer':uncer, 'bad':bad}
            return cost_dict

In [ ]:
# attempt_num=2000
MAX_ITER=1000
N = 1

for p in range(10):

    psize=int(10*(p+1))
    N = psize

    for number in range(10):
        a=[]        
        sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff = np.load('GD_each/'+str(psize)+'_'+str(number)+'_params.npy')


            
        foldername = str(psize)+'_'+str(number)        
        K = torch.from_numpy(np.load(foldername+'.npy')).to('cuda').type(dtype = torch.float)
        K1 = np.load(foldername+'.npy').astype('complex64')

        for i in range(0,np.shape(K1[0,0,:])[0]):
            for j in range(i+1,np.shape(K1[0,0,:])[0]):
                for k in range(j+1,np.shape(K1[0,0,:])[0]):
                      if (np.random.rand(1)[0]>(1-np.shape(K1[0,0,:])[0]/50/100)):
                        K1[i,j,k] = K[i,j,k]*(1.0+3.0*1j)
                        K1[i,k,j] = K1[i,j,k]
                        K1[j,i,k] = K1[i,j,k]
                        K1[j,k,i] = K1[i,j,k]
                        K1[k,i,j] = K1[i,j,k]
                        K1[k,j,i] = K1[i,j,k]
        K1_ten=torch.from_numpy(K1).to('cuda')

        attempt_num=500
        unity=torch.ones((N,attempt_num))
        zeros=torch.zeros((N,attempt_num))
        clip_percents=50
        population_size=7
        percentile = 50

        TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff,MAX_ITER, N, attempt_num, unity, zeros)
        interface = TGDInterface(TGD,attempt_num,percentile)

        first_params = np.array([np.log10(TGD.sigma), np.log10(TGD.dt), np.log10(TGD.zeta), np.log10(TGD.eps_coeff), np.log10(TGD.rho_thr_coeff), np.log10(TGD.gamma_init_coeff), np.log10(TGD.D_coeff)])
        min_boundary = np.array([0,-6,-4,-4,-4, -4,0])
        max_boundary = np.array([ 3, 0, 3, 3, 3, 3, 2])
        num_params = 7
        config = dict(controller_type='neural_net', 
                      training_type = 'differential_evolution',
                      first_params = first_params,
                      min_boundary = min_boundary, 
                      max_boundary = max_boundary,
                      num_params = num_params,
                      trust_region = 0.5,#0.5
                      max_num_runs = 1500,
                      num_training_runs = 2 * num_params,
                      log_level = 0, 
                      no_delay = False, 
                      max_num_runs_without_better_params = 150, 
                      population_size=10, 
                      mutation_scale = (0.5,1.0), 
                      cross_over_probability = 0.7, 
                      default_bad_cost = 1.05, 
                      default_bad_uncertainty = 0.01,
                      cost_has_noise = True)

        controller = mlc.create_controller(interface, **config)
        controller.optimize()
        clear_output(True)


        print('Best cut found:')

        mlv.show_all_default_visualizations(controller)
        sigma=10**controller.best_params[0]
        dt=10**controller.best_params[1]
        zeta=10**controller.best_params[2]
        eps_coeff=10**controller.best_params[3]
        rho_thr_coeff=10**controller.best_params[4]
        gamma_init_coeff=10**controller.best_params[5]
        D_coeff=10**controller.best_params[6]   

        attempt_num=2000
        params=[sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff]
        print(p, number)
        unity=torch.ones((N,attempt_num),  device = torch.device('cuda'))
        zeros=torch.zeros((N,attempt_num), device = torch.device('cuda'))
        TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros)
        H_opt1, spins1, phi_1, gamma=TGD.GD_eval()
        H_opt1_nump=H_opt1.cpu().numpy()
        plt.show()
        H_opt2, spins2, casda, asdasd=TGD.GD_eval_cc(phi_1, gamma, K1_ten,K)
        H_opt2_nump=H_opt2.cpu().numpy()
        while  np.sum(H_opt2_nump==2000)>0 or np.sum(H_opt1_nump==2000)>0:
            H_opt1, spins1, phi_1, gamma=TGD.GD_eval()
            H_opt1_nump=H_opt1.cpu().numpy()
            print(phi_1.cpu().numpy().max(), gamma.cpu().numpy().max())
            TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros)
            H_opt2, spins2, casda, asdas=TGD.GD_eval_cc(phi_1, gamma, K1_ten,K)
            H_opt2_nump=H_opt2.cpu().numpy()
        plt.hist(H_opt1_nump, color='blue', alpha=0.5)
        plt.hist(H_opt2_nump, color='red', alpha=0.5)
        plt.show()
        print(H_opt1_nump.mean(), H_opt2_nump.mean())
        print(H_opt1_nump.min(), H_opt2_nump.min())
        H_opt=H_opt1_nump*(H_opt1_nump<=H_opt2_nump)+H_opt2_nump*(H_opt1_nump>H_opt2_nump)
        spins = np.array(spins1.cpu()) * (H_opt1_nump<=H_opt2_nump).reshape(1,-1) + np.array(spins2.cpu()) *(H_opt1_nump>H_opt2_nump).reshape(1,-1)

        np.save('GD_each/'+str(psize)+'_'+str(number)+'_bit.npy', spins)
        np.save('GD_each/'+str(psize)+'_'+str(number)+'.npy', H_opt)
        np.save('GD_each/'+str(psize)+'_'+str(number)+'_params.npy', params)
